# 07 — Historical Weather Events & OGG Recovery Dataset

This notebook changes the unit of analysis from **individual flights** to **airport-time windows and weather-event episodes**.

Goals:
1. aggregate OGG operations into hourly bins;
2. measure cancellation, delay, severe-disruption, and throughput behavior through time;
3. identify Maui/OGG-relevant NOAA Storm Event episodes;
4. summarize weather severity during each episode;
5. estimate operational recovery time after an event;
6. save an event-level table for later recovery modeling and historical-similarity retrieval.

> Recovery is an operational proxy, not an official FAA airport reopening timestamp. We define recovery from observed BTS flight outcomes.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'data').exists() else cwd.parent
MERGED_FILE = ROOT / 'data/processed/ogg_flight_weather_storm_2020_2026.csv.gz'
STORM_FILE = ROOT / 'data/raw/incidents/hawaii_storm_events_2020_2026.csv.gz'
OUT_HOURLY = ROOT / 'data/processed/ogg_hourly_operations_2020_2025.csv.gz'
OUT_EVENTS = ROOT / 'data/processed/ogg_weather_event_recovery_2020_2025.csv'

print('Merged flight table:', MERGED_FILE, MERGED_FILE.exists())
print('Storm table:', STORM_FILE, STORM_FILE.exists())


## 1. Load flight-level merged data and storm events


In [ ]:
flights = pd.read_csv(MERGED_FILE, low_memory=False)
storms = pd.read_csv(STORM_FILE, low_memory=False)
flights['ogg_sched_dt'] = pd.to_datetime(flights['ogg_sched_dt'], errors='coerce')
flights = flights.dropna(subset=['ogg_sched_dt']).copy()

# Restrict recovery analysis to years with weather coverage.
flights = flights[flights['ogg_sched_dt'].dt.year <= 2025].copy()
print('Flights:', flights.shape)
print('Storm events:', storms.shape)
print('Range:', flights['ogg_sched_dt'].min(), 'to', flights['ogg_sched_dt'].max())


## 2. Build hourly OGG operational state
We aggregate scheduled flights into local OGG clock-hour bins. Empty hours are retained so recovery trajectories remain continuous.


In [ ]:
flights['hour'] = flights['ogg_sched_dt'].dt.floor('h')
flights['is_cancelled'] = pd.to_numeric(flights.get('Cancelled', 0), errors='coerce').fillna(0).gt(0).astype(int)
arr_delay = pd.to_numeric(flights.get('ArrDelayMinutes', flights.get('ArrDelay')), errors='coerce')
dep_delay = pd.to_numeric(flights.get('DepDelayMinutes', flights.get('DepDelay')), errors='coerce')
flights['delay_minutes'] = np.where(flights['direction'].eq('arrival'), arr_delay, dep_delay)
flights['is_delayed15'] = flights['delay_minutes'].ge(15).astype(int)
flights['is_severe'] = (flights['is_cancelled'].eq(1) | flights['delay_minutes'].ge(120)).astype(int)

agg = flights.groupby('hour').agg(
    scheduled_flights=('hour', 'size'),
    cancelled=('is_cancelled', 'sum'),
    delayed15=('is_delayed15', 'sum'),
    severe=('is_severe', 'sum'),
    mean_delay_min=('delay_minutes', 'mean'),
    median_delay_min=('delay_minutes', 'median'),
).reset_index()

full_hours = pd.DataFrame({'hour': pd.date_range(agg['hour'].min(), agg['hour'].max(), freq='h')})
hourly = full_hours.merge(agg, on='hour', how='left')
for c in ['scheduled_flights','cancelled','delayed15','severe']:
    hourly[c] = hourly[c].fillna(0).astype(int)

hourly['cancel_rate'] = np.where(hourly['scheduled_flights'] > 0, hourly['cancelled']/hourly['scheduled_flights'], np.nan)
hourly['delay15_rate'] = np.where(hourly['scheduled_flights'] > 0, hourly['delayed15']/hourly['scheduled_flights'], np.nan)
hourly['severe_rate'] = np.where(hourly['scheduled_flights'] > 0, hourly['severe']/hourly['scheduled_flights'], np.nan)
display(hourly.head())
print('Hourly rows:', len(hourly))


## 3. Add weather state to hourly operations
Weather observations already attached to flights in Notebook 03 are summarized by hour. This avoids a second raw-weather merge while preserving the conditions experienced around scheduled operations.


In [ ]:
weather_cols = [c for c in flights.columns if c.endswith('_num')]
print('Numeric weather columns:', weather_cols)

if weather_cols:
    weather_hour = flights.groupby('hour')[weather_cols].median().reset_index()
    hourly = hourly.merge(weather_hour, on='hour', how='left')

# Useful rolling operational context. Shift by one hour so baselines use only prior operations.
hourly['prior_6h_cancel_rate'] = hourly['cancelled'].rolling(6, min_periods=1).sum().shift(1) / hourly['scheduled_flights'].rolling(6, min_periods=1).sum().shift(1).replace(0, np.nan)
hourly['prior_6h_severe_rate'] = hourly['severe'].rolling(6, min_periods=1).sum().shift(1) / hourly['scheduled_flights'].rolling(6, min_periods=1).sum().shift(1).replace(0, np.nan)
hourly['prior_24h_cancel_rate'] = hourly['cancelled'].rolling(24, min_periods=1).sum().shift(1) / hourly['scheduled_flights'].rolling(24, min_periods=1).sum().shift(1).replace(0, np.nan)
hourly['prior_24h_severe_rate'] = hourly['severe'].rolling(24, min_periods=1).sum().shift(1) / hourly['scheduled_flights'].rolling(24, min_periods=1).sum().shift(1).replace(0, np.nan)
display(hourly.tail())


## 4. Parse and filter Maui/OGG-relevant storm events


In [ ]:
def parse_noaa_dt(series):
    s = series.astype(str).str.strip()
    # Storm Events commonly uses DD-MON-YY HH:MM:SS; mixed parsing is retained as fallback.
    parsed = pd.to_datetime(s, format='%d-%b-%y %H:%M:%S', errors='coerce')
    miss = parsed.isna()
    if miss.any():
        parsed.loc[miss] = pd.to_datetime(s.loc[miss], errors='coerce')
    return parsed

storms['begin_dt'] = parse_noaa_dt(storms['BEGIN_DATE_TIME'])
storms['end_dt'] = parse_noaa_dt(storms['END_DATE_TIME'])

text_cols = [c for c in ['CZ_NAME','CZ_NAME_STR','EVENT_TYPE','SOURCE','EPISODE_NARRATIVE','EVENT_NARRATIVE'] if c in storms.columns]
storm_text = storms[text_cols].fillna('').astype(str).agg(' '.join, axis=1).str.upper()
maui_mask = storm_text.str.contains(r'\bMAUI\b|KAHULUI|\bOGG\b', regex=True)
maui = storms[maui_mask & storms['begin_dt'].notna() & storms['end_dt'].notna()].copy()
maui = maui[maui['end_dt'] >= maui['begin_dt']].copy()
maui = maui[maui['begin_dt'].dt.year <= 2025].copy()

print('Maui/OGG-relevant event records:', len(maui))
display(maui[['EVENT_TYPE','begin_dt','end_dt'] + [c for c in ['CZ_NAME','CZ_NAME_STR'] if c in maui.columns]].head(20))


## 5. Collapse overlapping storm records into event episodes
NOAA may contain several records during the same broader weather episode. Records that overlap or begin within six hours of the previous record are combined.


In [ ]:
maui = maui.sort_values('begin_dt').reset_index(drop=True)
episodes = []
gap = pd.Timedelta(hours=6)

for idx, row in maui.iterrows():
    start, end = row['begin_dt'], row['end_dt']
    etype = str(row.get('EVENT_TYPE', 'Unknown'))
    if not episodes or start > episodes[-1]['end_dt'] + gap:
        episodes.append({'start_dt': start, 'end_dt': end, 'event_types': {etype}, 'storm_records': 1})
    else:
        episodes[-1]['end_dt'] = max(episodes[-1]['end_dt'], end)
        episodes[-1]['event_types'].add(etype)
        episodes[-1]['storm_records'] += 1

episodes = pd.DataFrame(episodes)
episodes['event_id'] = ['OGG_EVT_%03d' % (i+1) for i in range(len(episodes))]
episodes['event_types'] = episodes['event_types'].apply(lambda x: ' | '.join(sorted(x)))
episodes['duration_hours'] = (episodes['end_dt'] - episodes['start_dt']).dt.total_seconds()/3600
episodes['year'] = episodes['start_dt'].dt.year
episodes['month'] = episodes['start_dt'].dt.month
print('Collapsed episodes:', len(episodes))
display(episodes.head(20))


## 6. Event impact and recovery definition

For each episode we inspect **24 h before** through **72 h after**. The pre-event baseline is computed from scheduled-flight hours in the prior 24 h.

A post-event hour is considered operationally recovered when:
- it has scheduled flights;
- its rolling 3-hour cancellation rate is no more than `max(5%, baseline + 5 percentage points)`; and
- its rolling 3-hour severe-disruption rate is no more than `max(10%, baseline + 5 percentage points)`.

We require **three consecutive qualifying hours**. This is deliberately transparent and can later be sensitivity-tested.


In [ ]:
PRE_HOURS = 24
POST_HOURS = 72
CONSECUTIVE_RECOVERY_HOURS = 3

def weighted_rate(frame, num):
    den = frame['scheduled_flights'].sum()
    return frame[num].sum()/den if den > 0 else np.nan

def first_recovery_hour(post, cancel_limit, severe_limit):
    p = post.copy()
    sched3 = p['scheduled_flights'].rolling(3, min_periods=1).sum()
    p['cancel_3h'] = p['cancelled'].rolling(3, min_periods=1).sum()/sched3.replace(0, np.nan)
    p['severe_3h'] = p['severe'].rolling(3, min_periods=1).sum()/sched3.replace(0, np.nan)
    p['ok'] = (p['scheduled_flights'] > 0) & p['cancel_3h'].le(cancel_limit) & p['severe_3h'].le(severe_limit)
    run = p['ok'].rolling(CONSECUTIVE_RECOVERY_HOURS).sum().eq(CONSECUTIVE_RECOVERY_HOURS)
    if not run.any():
        return pd.NaT
    end_idx = run[run].index[0]
    pos = p.index.get_loc(end_idx)
    start_pos = pos - CONSECUTIVE_RECOVERY_HOURS + 1
    return p.iloc[start_pos]['hour']

rows = []
for ep in episodes.itertuples(index=False):
    start_hour = pd.Timestamp(ep.start_dt).floor('h')
    end_hour = pd.Timestamp(ep.end_dt).ceil('h')
    pre = hourly[(hourly['hour'] >= start_hour-pd.Timedelta(hours=PRE_HOURS)) & (hourly['hour'] < start_hour)]
    during = hourly[(hourly['hour'] >= start_hour) & (hourly['hour'] <= end_hour)]
    post = hourly[(hourly['hour'] > end_hour) & (hourly['hour'] <= end_hour+pd.Timedelta(hours=POST_HOURS))].copy()

    base_cancel = weighted_rate(pre, 'cancelled')
    base_severe = weighted_rate(pre, 'severe')
    cancel_limit = max(0.05, (base_cancel if pd.notna(base_cancel) else 0) + 0.05)
    severe_limit = max(0.10, (base_severe if pd.notna(base_severe) else 0) + 0.05)
    recovery_dt = first_recovery_hour(post, cancel_limit, severe_limit)
    recovery_hours = (recovery_dt-end_hour).total_seconds()/3600 if pd.notna(recovery_dt) else np.nan

    r = {
        'event_id': ep.event_id, 'start_dt': ep.start_dt, 'end_dt': ep.end_dt,
        'event_types': ep.event_types, 'storm_records': ep.storm_records,
        'duration_hours': ep.duration_hours, 'year': ep.year, 'month': ep.month,
        'pre_scheduled': pre['scheduled_flights'].sum(),
        'event_scheduled': during['scheduled_flights'].sum(),
        'event_cancelled': during['cancelled'].sum(),
        'event_delayed15': during['delayed15'].sum(),
        'event_severe': during['severe'].sum(),
        'baseline_cancel_rate': base_cancel,
        'baseline_severe_rate': base_severe,
        'event_cancel_rate': weighted_rate(during, 'cancelled'),
        'event_severe_rate': weighted_rate(during, 'severe'),
        'peak_hour_cancel_rate': during['cancel_rate'].max(),
        'peak_hour_severe_rate': during['severe_rate'].max(),
        'recovery_dt': recovery_dt, 'recovery_hours_after_event': recovery_hours,
        'recovered_within_72h': int(pd.notna(recovery_dt)),
    }
    for c in weather_cols:
        if c in during.columns:
            r[f'event_{c}_mean'] = during[c].mean()
            r[f'event_{c}_max'] = during[c].max()
            r[f'event_{c}_min'] = during[c].min()
    rows.append(r)

event_df = pd.DataFrame(rows)
print('Event-level rows:', event_df.shape)
display(event_df.head())


## 7. Inspect the most operationally disruptive historical episodes


In [ ]:
cols = ['event_id','start_dt','end_dt','event_types','event_scheduled','event_cancelled','event_cancel_rate','event_severe_rate','recovery_hours_after_event']
display(event_df.sort_values(['event_cancel_rate','event_severe_rate'], ascending=False)[cols].head(20).round(3))
print('Events with estimated recovery:', event_df['recovered_within_72h'].sum(), '/', len(event_df))
print('Median recovery hours:', event_df['recovery_hours_after_event'].median())


## 8. Airline-specific behavior during each episode
This table will later support statements such as which carriers historically cancelled more aggressively during comparable conditions.


In [ ]:
airline_col = next((c for c in ['Reporting_Airline','Operating_Airline','Marketing_Airline_Network'] if c in flights.columns), None)
airline_rows = []
if airline_col:
    for ep in episodes.itertuples(index=False):
        f = flights[(flights['ogg_sched_dt'] >= ep.start_dt) & (flights['ogg_sched_dt'] <= ep.end_dt)].copy()
        if f.empty:
            continue
        g = f.groupby(airline_col).agg(
            scheduled=('ogg_sched_dt','size'),
            cancelled=('is_cancelled','sum'),
            severe=('is_severe','sum'),
            mean_delay_min=('delay_minutes','mean'),
        ).reset_index()
        g['cancel_rate'] = g['cancelled']/g['scheduled']
        g['severe_rate'] = g['severe']/g['scheduled']
        g['event_id'] = ep.event_id
        airline_rows.append(g)
    airline_event = pd.concat(airline_rows, ignore_index=True) if airline_rows else pd.DataFrame()
    display(airline_event.sort_values('cancel_rate', ascending=False).head(20))
else:
    airline_event = pd.DataFrame()
    print('No airline identifier found.')


## 9. Save recovery datasets


In [ ]:
OUT_HOURLY.parent.mkdir(parents=True, exist_ok=True)
hourly.to_csv(OUT_HOURLY, index=False, compression='gzip')
event_df.to_csv(OUT_EVENTS, index=False)
print('Saved hourly operations:', OUT_HOURLY, hourly.shape)
print('Saved event recovery table:', OUT_EVENTS, event_df.shape)

if not airline_event.empty:
    airline_out = ROOT / 'data/processed/ogg_airline_event_behavior_2020_2025.csv'
    airline_event.to_csv(airline_out, index=False)
    print('Saved airline-event behavior:', airline_out, airline_event.shape)


## Next step
Notebook 08 can use this event table for **historical similar-event retrieval** and, if the number of usable episodes is sufficient, recovery-time modeling. Because NOAA event records are not official airport closure/reopening records, the recovery target here should remain labeled as an **observed operational recovery proxy**.
